In [ ]:
import logging
import traceback

from src.data_handle import json_handle
from src.embedding import generate_dense_embeddings_with_m3e, reduce_dimensionality
from storage.Faiss_storage import build_faiss_index

# 配置日志
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

dense_embeddings_reduced = None
pca_model = None
scaler_model = None
try:
    # 步骤 1：获取有效文本块
    valid_text_chunks = json_handle.batch_split()

    # 步骤 2：生成 m3e-base 稠密向量（替换原 OpenAI 向量生成）
    # 演示：取前 1000 条文本块（百万级可直接使用 valid_text_chunks）
    demo_text_chunks = valid_text_chunks[:1000]
    dense_embeddings = generate_dense_embeddings_with_m3e(demo_text_chunks)

    # 验证向量格式（确保与后续 PCA、FAISS 兼容）
    logger.info(f"m3e 向量验证：维度 {dense_embeddings.shape[1]}，数据类型 {dense_embeddings.dtype}")
    print(f"\n示例文本块：\n{demo_text_chunks[0][:200]}...")
    print(f"\n示例向量形状：{dense_embeddings.shape}")

    # 对稠密向量进行降维
    dense_embeddings_reduced, pca_model, scaler_model = reduce_dimensionality(
        dense_embeddings,
        variance_threshold=0.95  # 保留95%的语义方差，平衡效率与效果
    )

    # 构建百万级（演示10万条）FAISS索引
    faiss_index = build_faiss_index(dense_embeddings_reduced)

except ImportError as e:
    logger.critical(f"模块导入失败：{str(e)}")
    logger.debug(traceback.format_exc())
    exit(1)
except RuntimeError as e:
    logger.critical(f"运行时错误：{str(e)}")
    logger.debug(traceback.format_exc())
    exit(1)
except Exception as e:
    logger.critical(f"未知错误：{str(e)}")
    logger.debug(traceback.format_exc())
    exit(1)

In [ ]:
import logging

from src.data_handle import json_handle
from src.embedding import generate_dense_embeddings_with_m3e, reduce_dimensionality

# 配置日志
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")
logger = logging.getLogger(__name__)

dense_embeddings_reduced = None
pca_model = None
scaler_model = None

# 步骤 1：获取有效文本块
valid_text_chunks = json_handle.batch_split()

# 步骤 2：生成 m3e-base 稠密向量（替换原 OpenAI 向量生成）
# 演示：取前 1000 条文本块（百万级可直接使用 valid_text_chunks）
demo_text_chunks = valid_text_chunks[:256]
dense_embeddings = generate_dense_embeddings_with_m3e(demo_text_chunks)

# 验证向量格式（确保与后续 PCA、FAISS 兼容）
logger.info(f"m3e 向量验证：维度 {dense_embeddings.shape[1]}，数据类型 {dense_embeddings.dtype}")
print(f"\n示例文本块：\n{demo_text_chunks[0][:200]}...")
print(f"\n示例向量形状：{dense_embeddings.shape}")

# 对稠密向量进行降维
dense_embeddings_reduced, pca_model, scaler_model = reduce_dimensionality(
    dense_embeddings,
    variance_threshold=0.95  # 保留95%的语义方差，平衡效率与效果
)

In [ ]:
# 构建百万级（演示10万条）FAISS索引
from storage.Faiss_storage import build_faiss_index
faiss_index = build_faiss_index(dense_embeddings_reduced, save_path='faiss_million_index.index')

# 加载索引（后续检索可直接加载，无需重新构建）
# faiss_index = faiss.read_index(get_storage_path("faiss_fixed_pq_1.13.2.index"))

In [1]:
from src.application import EnterpriseRAGPipeline

# 方法1：使用新的类方法从本地存储直接加载（推荐）
try:
    print("正在从本地存储加载RAG组件...")
    rag_pipeline = EnterpriseRAGPipeline.from_local_storage(
        faiss_index_path="faiss_fixed_pq_1.13.2.index",  # 使用现有的索引文件
        top_k=5
    )

    # 测试查询（心理咨询场景）
    test_query = "和男朋友因为工资卡管理产生矛盾，感到不安怎么办？"
    rag_result = rag_pipeline.run(test_query)

    # 打印结构化结果（验证qwen-plus输出）
    print("=" * 80)
    print(f"查询状态：{rag_result['status']}")
    print(f"使用模型：{rag_result['qa_model']}")
    print(f"总处理耗时：{rag_result['total_processing_time_seconds']}秒")
    print(f"\n用户查询：{rag_result['query']}")
    print(f"\n回答结果：\n{rag_result['answer']}")
    print(f"\n检索到的上下文数量：{len(rag_result['contexts'])}")
    print(f"\n最相似上下文距离：{rag_result['distances'][0] if rag_result['distances'] else '无'}")
    print("=" * 80)
except Exception as e:
    logger.critical(f"RAG流水线初始化或运行失败：{str(e)}")
    print(f"错误详情：{str(e)}")
    exit(1)

# 方法2：或者使用独立的加载函数（备选方案）
# from application.local_loader import load_production_rag
# try:
#     print("正在使用加载器创建RAG流水线...")
#     rag_pipeline = load_production_rag(top_k=5)
#     
#     # 测试查询
#     test_query = "和男朋友因为工资卡管理产生矛盾，感到不安怎么办？"
#     rag_result = rag_pipeline.run(test_query)
#     
#     # 打印结果
#     print("=" * 80)
#     print(f"查询状态：{rag_result['status']}")
#     print(f"使用模型：{rag_result['qa_model']}")
#     print(f"总处理耗时：{rag_result['total_processing_time_seconds']}秒")
#     print(f"\n用户查询：{rag_result['query']}")
#     print(f"\n回答结果：\n{rag_result['answer']}")
#     print(f"\n检索到的上下文数量：{len(rag_result['contexts'])}")
#     print(f"\n最相似上下文距离：{rag_result['distances'][0] if rag_result['distances'] else '无'}")
#     print("=" * 80)
# except Exception as e:
#     logger.critical(f"RAG流水线初始化或运行失败：{str(e)}")
#     print(f"错误详情：{str(e)}")
#     exit(1)

D:\rag-project\rag_demo1\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-02-01 12:11:56.115 | INFO     | embedding.M3EEmbedding:_load_model:46 - 开始加载 m3e 模型：moka-ai/m3e-base
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 636.13it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: D:\rag-project\rag_demo1\resource_package/models\moka-ai/m3e-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-02-01 12:11:56.605 | SUCCESS  | embedding.M3EEmbedding:_load_model:53 - m3e 模型加载成功，运行设备：cpu，向量维度：768
2026-02-01 12:11:57.338 | WARNING  | application.a

正在从本地存储加载RAG组件...


NameError: name 'logger' is not defined